# Module 7: Two Models on One GPU

In Module 6 you shrank a model with quantization and won back GPU memory. Now you spend that headroom: you put two models on one card and route each request to the right one. Not every prompt needs your biggest model. A classifier or a quick rewrite can go to a small fast model, while a hard reasoning task goes to a larger one. [vLLM](https://docs.vllm.ai) serves exactly one model per process, so two models means two vLLM servers sharing one GPU. This module stands both up, puts a content router in front, and watches them contend for the same KV cache.

## Learning objectives
- Explain why two models on one GPU means two vLLM processes, not one
- Stand up a fast and a smart backend from one manifest and confirm both answer
- Route a prompt to the fast or smart model with a plain-Python content router
- Drive load at both backends at once and read each one's `kv_cache_usage_perc`
- See co-location as a tradeoff: one GPU bill and simpler ops, paid for in headroom
- Decide when to split a card and when a model has outgrown its slice

## Prerequisites
- Finished Module 6, so you know quantization buys the footprint that makes two models fit
- Both backends deployed from `manifests/two-models.yaml`, reachable as `vllm-fast` and `vllm-smart`
- GPU time-slicing (or MPS) enabled on the platform, which the workshop platform does for you
- About 12 minutes

References: [vLLM engine args](https://docs.vllm.ai/en/latest/serving/engine_args.html) &middot; [NVIDIA GPU time-slicing](https://docs.nvidia.com/datacenter/cloud-native/gpu-operator/latest/gpu-sharing.html) &middot; [agentgateway routing](https://agentgateway.dev/) &middot; [Akamai Cloud GPUs](https://www.linode.com/products/gpu/)

## Two models on one GPU design basics

vLLM serves one model per process. To run two models you run two vLLM servers. The manifest in `manifests/two-models.yaml` does exactly that: a `vllm-fast` Deployment with a small model and a `vllm-smart` Deployment with a larger one. Each one caps `--gpu-memory-utilization` at `0.45`, so the two sets of weights plus both KV caches fit inside one card.

- **Two processes, one card.** Sharing a single physical GPU between two pods needs the cluster to allow it, through NVIDIA time-slicing or MPS. With it on, both pods land on the same card. Without it, the second Deployment asks for a whole GPU it cannot get and stays `Pending`.
- **A content router.** A small rule sends simple prompts to the fast model and hard prompts to the smart one. In production a gateway does this so clients send one `model` field. Here the router is plain Python so you can read the logic.
- **Shared KV cache.** Each model gets a `0.45` slice of the card, and each slice holds that model's weights plus its share of the KV cache. When both are busy they pull from the same finite card, so heavy load on one shows up as pressure on the other.

![One GPU time-sliced between vllm-fast and vllm-smart, with a content router sending simple prompts to fast and hard prompts to smart while both contend for the shared KV cache](images/07_two_models_one_gpu_architecture.png)

## 1. Setup

This module talks to two vLLM endpoints directly, so it needs only the OpenAI client. Install it. We reinstall here so this notebook stands on its own.

In [ ]:
%pip install -q "openai>=1.40"

## 2. Configure both backends

Two models means two endpoints, so you build one client per backend. `FAST_HOST`, `SMART_HOST`, `FAST_MODEL`, and `SMART_MODEL` read from your environment with sensible defaults, so override them only if your Service names or model ids differ. The metrics URL for each is the same host with `/metrics` in place of `/v1`, the same derivation the other labs use.

In [ ]:
# Setup: build one client per backend. Each model is its own endpoint.
import os, sys, time, threading
from concurrent.futures import ThreadPoolExecutor
sys.path.insert(0, os.path.abspath(".."))

from openai import OpenAI
from common import metrics

# The two backends. Override with env if your Service names differ.
FAST_HOST = os.environ.get("FAST_HOST", "http://vllm-fast:8000/v1")
SMART_HOST = os.environ.get("SMART_HOST", "http://vllm-smart:8000/v1")
FAST_MODEL = os.environ.get("FAST_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
SMART_MODEL = os.environ.get("SMART_MODEL", "Qwen/Qwen2.5-3B-Instruct")

fast_client = OpenAI(base_url=FAST_HOST, api_key="not-needed")
smart_client = OpenAI(base_url=SMART_HOST, api_key="not-needed")

# Metrics URLs are the same host with /metrics instead of /v1.
def metrics_url_for(base):
    root = base.rstrip("/")
    if root.endswith("/v1"):
        root = root[:-3]
    return root + "/metrics"

fast_metrics = metrics_url_for(FAST_HOST)
smart_metrics = metrics_url_for(SMART_HOST)

print("fast :", FAST_HOST, "->", FAST_MODEL)
print("smart:", SMART_HOST, "->", SMART_MODEL)

**What you should see:** the two backends and their models, one client pointed at the fast model and one at the smart model. If a host is wrong here, every cell below fails the same way, so confirm both lines first.

## 3. Confirm both models answer

Send the same prompt to each backend and time it. The fast model should reply quicker; the smart model should give a fuller answer. That gap, speed against quality, is the tradeoff you route on. If either call errors, that backend is not up yet, which usually means time-slicing is off and the second Deployment is still `Pending`.

In [ ]:
# Requires both vLLM endpoints live.
# Ask both models the same question and time each.
question = "Explain in two sentences why batching helps GPU inference throughput."

for name, client, model in [("fast", fast_client, FAST_MODEL), ("smart", smart_client, SMART_MODEL)]:
    start = time.time()
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": question}],
        max_tokens=120, temperature=0.0,
    )
    print(f"[{name}] {time.time()-start:.2f}s")
    print(resp.choices[0].message.content.strip(), "\n")

**What you should see:** two answers. The fast model returns sooner, the smart model usually gives the richer reply. Both ran on the same GPU.

## 4. Route between them

Routing is just a decision about which backend gets the request. The rule below is deliberately plain: short, factual asks go to fast, anything longer or flagged hard goes to smart. You can see the whole logic, which is the point of writing it raw before reaching for a gateway. Swap in your own rule later, a token count, a classifier, or an explicit `model` field, and a production setup moves this into a gateway so clients send one `model` field and the gateway dispatches.

In [ ]:
# A content-based router. Returns (client, model_id, label) for a prompt.
HARD_HINTS = ("explain", "why", "design", "compare", "reason", "step by step")

def route(prompt):
    text = prompt.lower()
    is_hard = len(prompt.split()) > 20 or any(h in text for h in HARD_HINTS)
    if is_hard:
        return smart_client, SMART_MODEL, "smart"
    return fast_client, FAST_MODEL, "fast"

def ask(prompt, max_tokens=120):
    client, model, label = route(prompt)
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens, temperature=0.0,
    )
    return label, resp.choices[0].message.content.strip()

for p in ["What is 2+2?", "Compare continuous batching and static batching and explain the tradeoffs."]:
    label, answer = ask(p)
    print(f"[routed to {label}] {p}")
    print(answer[:160], "\n")

**What you should see:** the short arithmetic question routed to the fast model and the longer comparison routed to the smart one. The routing decision is printed so you can confirm the rule fired as expected.

## 5. Watch them contend

Now the point of the module. Drive load at both models at once and read each backend's `kv_cache_usage_perc`. Because they share one card, hammering the smart model eats memory the fast model also needs. You will see both caches under pressure even though the requests went to different endpoints. That contention is a first-class signal: it tells you the split is too tight, or that one model deserves more of the card.

In [ ]:
# Requires both endpoints live.
# Hammer both models concurrently and sample each one's KV cache usage.
peak = {"fast": 0.0, "smart": 0.0}
stop = threading.Event()

def sample():
    while not stop.is_set():
        for label, url in [("fast", fast_metrics), ("smart", smart_metrics)]:
            try:
                s = metrics.snapshot(url)
                peak[label] = max(peak[label], s["vllm:gpu_cache_usage_perc"])
            except Exception:
                pass
        time.sleep(0.25)

def load(client, model, n=16):
    def one(_):
        client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": "Write a long paragraph about Kubernetes."}],
            max_tokens=200, temperature=0.0,
        )
    with ThreadPoolExecutor(max_workers=n) as pool:
        list(pool.map(one, range(n)))

t = threading.Thread(target=sample, daemon=True); t.start()
# Load both at the same time so they compete for the shared card.
with ThreadPoolExecutor(max_workers=2) as pool:
    f1 = pool.submit(load, fast_client, FAST_MODEL, 16)
    f2 = pool.submit(load, smart_client, SMART_MODEL, 16)
    f1.result(); f2.result()
stop.set(); t.join(timeout=2)

print(f"peak KV usage  fast : {peak['fast']*100:.0f}%")
print(f"peak KV usage  smart: {peak['smart']*100:.0f}%")

**What you should see:** both backends report high peak KV cache usage during the simultaneous load. Each model has only its `0.45` slice, so each fills its own cache quickly. If you load one at a time, that one has its slice to itself but still cannot borrow the other model's half. That is the cost of co-location: simpler ops and one GPU bill, paid for with less headroom per model.

## 6. When to split a card and when not to

Co-locating two models makes sense when both are small, neither is constantly saturated, and you want one card to serve mixed traffic. It stops making sense when either model needs the whole card under peak load, because then the `0.45` cap throttles it. The honest rule: split the GPU when traffic is bursty and mixed, give a model its own card when it is steadily busy. The metrics in section 5 tell you which case you are in.

## Things to know

- **One model per process.** vLLM loads exactly one model per server, so N models is N processes. There is no single-process multi-model mode. Co-location is process co-location on a shared card.
- **Time-slicing is a platform setting.** Two pods land on one physical GPU only when the cluster enables NVIDIA time-slicing or MPS. Without it the second Deployment asks for a whole GPU and stays `Pending`. The workshop platform turns this on.
- **The gauge was renamed.** vLLM's V1 engine renamed `gpu_cache_usage_perc` to `kv_cache_usage_perc`. `common/metrics.py` `snapshot()` returns the value under both keys, so reading either name works no matter which engine the server runs.
- **The manifest is deliberate.** `enableServiceLinks: false` stops a Service named `vllm` from injecting a `VLLM_PORT` env var that vLLM would misread and crash on. `strategy: Recreate` avoids a rolling-update deadlock where the new pod cannot schedule until the old one frees its GPU slice. The `$(FAST_MODEL)` and `$(SMART_MODEL)` args are valid: Kubernetes expands them from the container's own `env`.

> NOTE: If a call in section 3 errors with a connection refused, the backend is not ready. Check that time-slicing is on and that neither Deployment is stuck `Pending`.

## Try it yourself

**Change the routing rule.** Edit `HARD_HINTS` or the word-count threshold and watch where each prompt lands. **Stretch:** route on `len(prompt)` in characters instead of words and see which prompts move.

**Shift the split.** In `manifests/two-models.yaml`, give the smart model more of the card (raise its `--gpu-memory-utilization`, lower the fast model's to match) and re-run section 5. The smart model should hold more headroom under load.

**Load one model only.** Run the contention cell against just the smart backend and compare its peak KV usage to the both-at-once run. The gap is the headroom co-location costs you.

In [ ]:
# Change the threshold or the hints, then run the cell.
HARD_HINTS = ("explain", "why", "design", "compare", "reason", "step by step")
WORD_THRESHOLD = 20   # prompts longer than this go to the smart model

def route_demo(prompt):
    text = prompt.lower()
    is_hard = len(prompt.split()) > WORD_THRESHOLD or any(h in text for h in HARD_HINTS)
    return "smart" if is_hard else "fast"

for p in ["What is 2+2?", "Summarize this in one line.", "Why does batching raise throughput?"]:
    print(f"[{route_demo(p)}] {p}")

## Summary

- Two models on one GPU means two vLLM processes sharing one card, each capped at a `0.45` memory slice so both fit.
- The card is shared only when the platform enables time-slicing or MPS; without it the second Deployment stays `Pending`.
- A plain-Python content router sends simple prompts to the fast model and hard prompts to the smart one, the same job a production gateway does.
- Under simultaneous load both backends fill their `kv_cache_usage_perc`, because they pull from the same finite card. Co-location trades headroom for one GPU bill and simpler ops.
- The metrics tell you when a model has outgrown its slice and deserves its own card.

## Next

**Module 8: Benchmark and Evaluate.** You have measured throughput, tuning, footprint, and now contention. Next you pull it all into a structured sweep, turn throughput into cost per million tokens, score quality on a real evaluation set, and write down a defensible operating point.